<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day4/notebooks/1_llm_annotation_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 4 — LLM Annotation & Distillation  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible answer
plus a short comment. LLM outputs vary between runs, so your exact numbers will differ.

> **Turn on the GPU!** *Runtime → Change runtime type → T4 GPU.*


## 0. Setup

In [ ]:
!pip install transformers datasets accelerate -q

import torch
import pandas as pd
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())

## 1. Loading a local open LLM

In [ ]:
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto",
)
print("LLM loaded and running locally!")

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "In one sentence, what is computational text analysis?"},
]
output = llm(messages, max_new_tokens=80)
print(output[0]["generated_text"][-1]["content"])

## 2. Annotation, take 1: zero-shot

In [ ]:
from datasets import load_dataset

bills = load_dataset("dreamproit/bill_labels_us", split="train").to_pandas()
bills = bills.rename(columns={"title": "text"})[["text", "policy_area"]].dropna()

AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
bills = bills[bills["policy_area"].isin(AREAS)]

sample = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 15), random_state=42)
).reset_index(drop=True)

print(f"Annotating {len(sample)} bill titles across {len(AREAS)} policy areas.")
sample.head()

In [ ]:
def annotate_zero_shot(text):
    system = (
        "You are an expert political science coder. "
        "Classify each bill title into exactly ONE of these policy areas:\n"
        "- Health\n- Education\n- Taxation\n- Armed Forces and National Security\n"
        "Reply with ONLY the policy area name, nothing else."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Bill title: {text}"},
    ]
    out = llm(messages, max_new_tokens=15, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

for text in sample["text"].head(3):
    print(f"'{text}'\n  -> {annotate_zero_shot(text)}\n")

> **✏️ Exercise 1**
>
> Run `annotate_zero_shot` on three bill titles of your own, including an ambiguous one.


In [ ]:
# ✅ Solution
my_titles = [
    "A bill to lower the cost of insulin for diabetics.",         # Health
    "An act to provide tax credits for college tuition.",         # Taxation OR Education?
    "A bill to fund medical care at veterans' military hospitals.", # Health OR Armed Forces?
]
for t in my_titles:
    print(f"'{t}'\n  -> {annotate_zero_shot(t)}\n")

# Comment: the clear title (insulin -> Health) is easy. The ambiguous ones are revealing:
# "tax credits for college tuition" genuinely straddles Taxation and Education, and
# "medical care at military hospitals" straddles Health and Armed Forces. The model has to
# pick one, and which it picks can feel arbitrary. Real codebooks handle this with explicit
# tie-breaking rules ("if a bill concerns both X and Y, code as X") — which you'd add to the
# prompt. Ambiguity is a property of the TASK, not a failure of the model.

## 3. The crucial step: validate against the gold standard

In [ ]:
sample["llm_label"] = sample["text"].apply(annotate_zero_shot)

def normalize(label):
    label = label.strip()
    for area in AREAS:
        if area.lower() in label.lower():
            return area
    return label

sample["llm_clean"] = sample["llm_label"].apply(normalize)
sample[["text", "policy_area", "llm_clean"]].head(10)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["policy_area"], sample["llm_clean"])
print(f"LLM agreement with human coders: {acc:.3f}\n")
print(classification_report(sample["policy_area"], sample["llm_clean"],
                            labels=AREAS, zero_division=0))

> **✏️ Exercise 2**
>
> Look at the disagreements. Are they reasonable or errors?


In [ ]:
# ✅ Solution
disagreements = sample[sample["policy_area"] != sample["llm_clean"]]
print(f"{len(disagreements)} disagreements out of {len(sample)}:\n")
for _, row in disagreements.iterrows():
    print(f"  '{row['text']}'")
    print(f"     human: {row['policy_area']}  |  LLM: {row['llm_clean']}\n")

# Comment: inspect each one. Many disagreements are on GENUINELY ambiguous titles — a bill
# about, say, military family education could reasonably be Education OR Armed Forces, and the
# human coder and the LLM simply chose differently. These aren't really "errors" so much as
# boundary cases where reasonable coders disagree (human coders disagree with each other too!).
# A smaller number may be true LLM mistakes. This qualitative review is ESSENTIAL: a raw
# accuracy number hides whether the model is failing badly or just splitting hairs on hard cases.

## 4. Improving annotation: few-shot prompting

In [ ]:
def annotate_few_shot(text):
    system = (
        "You are an expert political science coder. Classify each bill title into exactly "
        "ONE of: Health, Education, Taxation, Armed Forces and National Security. "
        "Reply with ONLY the policy area name."
    )
    examples = [
        ("A bill to fund cancer research at national institutes.", "Health"),
        ("An act to reduce student loan interest rates.", "Education"),
        ("A bill to adjust corporate tax brackets.", "Taxation"),
        ("An act to modernize the air force fleet.", "Armed Forces and National Security"),
    ]
    messages = [{"role": "system", "content": system}]
    for ex_text, ex_label in examples:
        messages.append({"role": "user", "content": f"Bill title: {ex_text}"})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": f"Bill title: {text}"})
    out = llm(messages, max_new_tokens=15, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

sample["llm_fewshot"] = sample["text"].apply(lambda t: normalize(annotate_few_shot(t)))
acc_few = accuracy_score(sample["policy_area"], sample["llm_fewshot"])
print(f"Zero-shot agreement: {acc:.3f}")
print(f"Few-shot agreement:  {acc_few:.3f}")

> **✏️ Exercise 3**
>
> Did few-shot help? Try harder / edge-case examples instead of obvious ones.


In [ ]:
# ✅ Solution
def annotate_few_shot_hard(text):
    system = (
        "You are an expert political science coder. Classify each bill title into exactly "
        "ONE of: Health, Education, Taxation, Armed Forces and National Security. "
        "Reply with ONLY the policy area name."
    )
    # EDGE-CASE examples that demonstrate the tricky boundaries
    examples = [
        ("A bill to fund mental health services on military bases.", "Health"),      # not Armed Forces
        ("An act to give tax deductions for teacher classroom expenses.", "Taxation"), # not Education
        ("A bill to provide tuition assistance to veterans.", "Education"),            # not Armed Forces
        ("An act to tax sugary drinks to fund health programs.", "Taxation"),          # not Health
    ]
    messages = [{"role": "system", "content": system}]
    for ex_text, ex_label in examples:
        messages.append({"role": "user", "content": f"Bill title: {ex_text}"})
        messages.append({"role": "assistant", "content": ex_label})
    messages.append({"role": "user", "content": f"Bill title: {text}"})
    out = llm(messages, max_new_tokens=15, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

sample["llm_hard"] = sample["text"].apply(lambda t: normalize(annotate_few_shot_hard(t)))
acc_hard = accuracy_score(sample["policy_area"], sample["llm_hard"])
print(f"Zero-shot:           {acc:.3f}")
print(f"Few-shot (obvious):  {acc_few:.3f}")
print(f"Few-shot (edge-case): {acc_hard:.3f}")

# Comment: edge-case examples often help MORE than obvious ones, because they teach the model
# the specific boundary rules it would otherwise get wrong (e.g. "a tax to fund health is
# Taxation, not Health"). This mirrors how you'd train a human coder — you don't waste time on
# easy cases, you drill the confusing ones. Choosing good few-shot examples is a real skill.

## 5. The payoff: distillation

In [ ]:
train_pool = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 60), random_state=1)
).reset_index(drop=True)

print(f"LLM is labeling {len(train_pool)} titles (the teacher step)... this takes a few minutes.")
train_pool["llm_label"] = train_pool["text"].apply(lambda t: normalize(annotate_few_shot(t)))
train_pool = train_pool[train_pool["llm_label"].isin(AREAS)].reset_index(drop=True)
print("Usable LLM-labeled examples:", len(train_pool))

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.model_selection import train_test_split

label2id = {a: i for i, a in enumerate(AREAS)}
id2label = {i: a for a, i in label2id.items()}
train_pool["label"] = train_pool["llm_label"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tok(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

tr, te = train_test_split(train_pool, test_size=0.2, random_state=42, stratify=train_pool["label"])
train_ds = Dataset.from_pandas(tr).map(tok, batched=True)
test_ds = Dataset.from_pandas(te).map(tok, batched=True)

student = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(AREAS),
    id2label=id2label, label2id=label2id,
)
print("Student model ready.")

In [ ]:
args = TrainingArguments(
    output_dir="./distilled",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="no",
    logging_steps=10,
    report_to="none",
)
trainer = Trainer(model=student, args=args, train_dataset=train_ds)
trainer.train()
print("Distillation complete.")

In [ ]:
import numpy as np

student_clf = pipeline("text-classification", model=student, tokenizer=tokenizer,
                       device=0 if device=="cuda" else -1)

def student_predict(text):
    return student_clf(text, truncation=True, max_length=32)[0]["label"]

sample["student_label"] = sample["text"].apply(student_predict)
acc_student = accuracy_score(sample["policy_area"], sample["student_label"])

print("Agreement with HUMAN gold standard:")
print(f"  LLM (teacher):        {acc_few:.3f}")
print(f"  DistilBERT (student): {acc_student:.3f}")

> **✏️ Exercise 4**
>
> Time the LLM vs the distilled BERT on 10 titles.


In [ ]:
# ✅ Solution
import time

titles_10 = sample["text"].head(10).tolist()

start = time.time()
for t in titles_10:
    annotate_few_shot(t)
llm_time = time.time() - start

start = time.time()
for t in titles_10:
    student_predict(t)
bert_time = time.time() - start

print(f"LLM (teacher):        {llm_time:.2f}s for 10 titles")
print(f"DistilBERT (student): {bert_time:.3f}s for 10 titles")
print(f"\nThe student is roughly {llm_time/bert_time:.0f}x faster.")

# Comment: the distilled BERT is typically ONE-TO-TWO ORDERS OF MAGNITUDE faster than the LLM.
# On a corpus of 100,000 documents, that is the difference between minutes and days — and the
# BERT needs no GPU to run, no API, and gives identical results every time. THIS is the
# practical case for distillation: you pay the slow LLM cost once, on a subset, then run the
# fast student on everything forever.

## 6. Doing this for real: Ollama on your own machine

*(See the student notebook — the Ollama workflow is explanatory, no code to run in Colab.)*


## Wrap-up

That's the full solution set — and the end of the course.

### Optional challenge

Train a second DistilBERT on **human** labels and compare to the LLM-label-trained one.


In [ ]:
# ✅ Solution — how much did using LLM labels (vs human) cost us?
# Train an identical student, but on the HUMAN policy_area labels for the same titles.
train_pool["human_label_id"] = train_pool["policy_area"].map(label2id)

tr_h, te_h = train_test_split(train_pool, test_size=0.2, random_state=42,
                              stratify=train_pool["human_label_id"])

def tok_h(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

# Relabel the 'label' column with human labels for this run
tr_h = tr_h.rename(columns={"label": "_llm", "human_label_id": "label"})
train_ds_h = Dataset.from_pandas(tr_h).map(tok_h, batched=True)

student_h = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(AREAS), id2label=id2label, label2id=label2id)

args_h = TrainingArguments(output_dir="./distilled_human", num_train_epochs=3,
                           per_device_train_batch_size=16, learning_rate=2e-5,
                           eval_strategy="no", logging_steps=10, report_to="none")
Trainer(model=student_h, args=args_h, train_dataset=train_ds_h).train()

clf_h = pipeline("text-classification", model=student_h, tokenizer=tokenizer,
                 device=0 if device=="cuda" else -1)
sample["student_human"] = sample["text"].apply(
    lambda t: clf_h(t, truncation=True, max_length=32)[0]["label"])
acc_student_human = accuracy_score(sample["policy_area"], sample["student_human"])

print(f"Student trained on LLM labels:   {acc_student:.3f}")
print(f"Student trained on HUMAN labels: {acc_student_human:.3f}")
print(f"Cost of using LLM labels:        {acc_student_human - acc_student:+.3f}")

# Comment: the human-label student usually scores a bit higher — that gap is the PRICE of the
# teacher's imperfections (the LLM's labels contain some errors, which the student inherits).
# The key question for your research: is that gap small enough to be worth the enormous savings
# in human labeling effort? Often yes — but you should measure it and report it, not assume it.